In [1]:
import pandas as pd
import ast
import os

def extract_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
import joblib
import glob
import time
from tqdm import tqdm
from orcapy import ORCA
from Bio import SeqIO
import multiprocessing

def find_oric(acc_n, result_dir, que):
    import warnings
    warnings.filterwarnings("ignore")
    os.makedirs(f'{result_dir}/{acc_n}', exist_ok=True)
    handle = open(f'/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/data/{acc_n}/genomic.gbff')
    acc_record = SeqIO.parse(handle, 'genbank')
    for seq_record in acc_record:
        if glob.glob(f'{result_dir}/{acc_n}/{seq_record.id}.csv'):
            continue
        file_name = f'{result_dir}/{acc_n}/{seq_record.id}.gbff'
        ncl_file = open(file_name, 'w+')
        seq_record.description = ''
        SeqIO.write(seq_record, ncl_file, 'genbank')
        ncl_file.close()
        orics = {'predictions': [], 'Z_scores': [], 'G_scores': [], 'D_scores': [], 'oriC_middles': []}
        try:
            orca = ORCA.from_gbk(file_name, model=model)
            orca.find_oriCs(show_info=False, plot_path=None)
            for j in range(len(orca.oriCs)):
                orics['predictions'].append(orca.oriCs[j].decision)
                orics['Z_scores'].append(orca.oriCs[j].z_score)
                orics['G_scores'].append(orca.oriCs[j].g_score)
                orics['D_scores'].append(orca.oriCs[j].d_score)
                orics['oriC_middles'].append(orca.oriCs[j].middle)
        except:
            pass
        os.system(f'rm {file_name}')
        result = pd.DataFrame(orics)
        result.to_csv(f'{result_dir}/{acc_n}/{seq_record.id}.csv', index=False)
    que.put(1)

model = joblib.load("/home/zhongshitong/ORCA_RFC_model_1_4_0.pkl.gz")
for genus_name in keep_genus:
    org_data_n = all_data[all_data['genus_clean'].str.contains(genus_name, na=False)].reset_index(drop=True)
    result_dir = f'/active-data/analysis_results/chr_pla/genus/statistics_records/{genus_name}/Ori_finder/oriC_ORCA'

    manager = multiprocessing.Manager()
    que = manager.Queue()
    
    par = 32
    tot = len(org_data_n)
    pool = multiprocessing.Pool(par)

    for i in org_data_n.index:
        acc_n = org_data_n['accession'][i]
        pool.apply_async(find_oric, (acc_n, result_dir, que))

    pool.close()
    
    count = 0
    with tqdm(total = tot, desc=f'{genus_name}({tot})', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        while True:
            time.sleep(0.001)
            if not que.empty():
                value = que.get(True)
                count += 1
                pbar.update(1)
                if count == tot:
                    break
            else:  
                continue
    
    pool.join()

Escherichia(4204): 100%|████████████████████████████████████████| 4.20k/4.20k [43:02<00:00, 1.63B/s]
Klebsiella(3554): 100%|███████████████████████████████████████| 3.55k/3.55k [1:05:17<00:00, 1.10s/B]
Staphylococcus(2423): 100%|█████████████████████████████████████| 2.42k/2.42k [24:47<00:00, 1.63B/s]
Pseudomonas(2343): 100%|██████████████████████████████████████| 2.34k/2.34k [1:00:03<00:00, 1.54s/B]
Bacillus(1976): 100%|████████████████████████████████████████| 1.98k/1.98k [22:54:32<00:00, 41.8s/B]
Salmonella(1853): 100%|█████████████████████████████████████████| 1.85k/1.85k [19:45<00:00, 1.56B/s]
Streptococcus(1599): 100%|██████████████████████████████████████| 1.60k/1.60k [05:07<00:00, 5.20B/s]
Streptomyces(1359): 100%|███████████████████████████████████████| 1.36k/1.36k [23:19<00:00, 1.03s/B]
Acinetobacter(1234): 100%|██████████████████████████████████████| 1.23k/1.23k [08:29<00:00, 2.42B/s]
Helicobacter(416): 100%|████████████████████████████████████████████| 416/416 [01:09<00:00,